In [9]:
%ls

2           catboost_info/  include/  main-work-folder/  pyvenv.cfg
artifacts/  data/           lib/      mlflow.db          share/
bin/        etc/            lib64@    mlruns/


In [10]:
import os
# Ищем ALS модель / факторы
for root, dirs, files in os.walk('./artifacts'):
    for f in files:
        path = os.path.join(root, f)
        if any(kw in f.lower() for kw in ['als', 'implicit', 'factor', 'embed', 'model']):
            size_mb = os.path.getsize(path) / 1024 / 1024
            print(f"{path} ({size_mb:.1f} MB)")

# Также проверим, можно ли поставить implicit
try:
    import implicit
    print(f"\nimplicit version: {implicit.__version__}")
except ImportError:
    print("\nimplicit not installed. Try: pip install implicit")


./artifacts/als_model.pkl (3.7 MB)
./artifacts/als_model_data.pkl (4.5 MB)
./artifacts/v3_user_factors.parquet (1.6 MB)
./artifacts/v3_item_factors.parquet (6.3 MB)
./artifacts/v3_item_factors_normed.npy (4.7 MB)

implicit version: 0.7.3


In [ ]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
import pickle
import warnings
warnings.filterwarnings('ignore')

import implicit
from implicit.als import AlternatingLeastSquares

history = pd.read_parquet('artifacts/history_interactions.parquet')
item_feat_mh = pd.read_parquet('artifacts/item_features_multihot.parquet')

print(f"History: {history.shape}")

# ============================================================
# 1. ALS 128 FACTORS
# ============================================================
user_ids = history['steamid'].unique()
all_candidate_items = np.union1d(
    pd.read_parquet('artifacts/train_reranker_base.parquet')['appid'].unique(),
    pd.read_parquet('artifacts/test_reranker_base.parquet')['appid'].unique()
)
all_items = np.union1d(history['appid'].unique(), all_candidate_items)

user2idx = {u: i for i, u in enumerate(user_ids)}
item2idx = {a: i for i, a in enumerate(all_items)}
idx2user = {i: u for u, i in user2idx.items()}
idx2item = {i: a for a, i in item2idx.items()}

n_users, n_items = len(user2idx), len(item2idx)
print(f"Matrix: {n_users} × {n_items}")

rows = history['steamid'].map(user2idx).values
cols = history['appid'].map(item2idx).values
confidence = (history['target'].values * 2.0 + np.log1p(history['playtime_forever'].values) * 0.5).astype(np.float32)
confidence = np.clip(confidence, 1.0, None)

user_item_matrix = csr_matrix((confidence, (rows, cols)), shape=(n_users, n_items), dtype=np.float32)

N_FACTORS = 128
print(f"\nTraining ALS with {N_FACTORS} factors...")
als_model = AlternatingLeastSquares(
    factors=N_FACTORS,
    regularization=0.01,
    iterations=30,
    random_state=42,
    use_gpu=False,
)
als_model.fit(user_item_matrix)
print("Done!")

user_factors = als_model.user_factors  # (n_users, 128)
item_factors = als_model.item_factors  # (n_items, 128)
print(f"User factors: {user_factors.shape}, Item factors: {item_factors.shape}")

# Normalized
from numpy.linalg import norm
item_norms = norm(item_factors, axis=1, keepdims=True).clip(1e-8)
item_factors_normed = item_factors / item_norms
user_norms = norm(user_factors, axis=1, keepdims=True).clip(1e-8)
user_factors_normed = user_factors / user_norms

# ============================================================
# 2. PRECOMPUTE USER CENTROIDS + DIVERSE TOP-K
# ============================================================
print("\nComputing user centroids and similarity profiles...")

binary_matrix = (user_item_matrix > 0).astype(np.float32)

user_centroids = np.zeros((n_users, N_FACTORS), dtype=np.float32)
user_top_by_conf = {}     # top-10 by confidence (=engagement)
user_taste_std = np.zeros(n_users, dtype=np.float32)  # taste diversity

for uid in range(n_users):
    played_idx = binary_matrix[uid].nonzero()[1]
    if len(played_idx) == 0:
        continue
    
    weights = user_item_matrix[uid, played_idx].toarray().flatten()
    weights_norm = weights / weights.sum()
    
    centroid = (item_factors_normed[played_idx] * weights_norm[:, None]).sum(axis=0)
    c_norm = norm(centroid)
    if c_norm > 1e-8:
        centroid = centroid / c_norm
    user_centroids[uid] = centroid
    
    # Top-10 by confidence
    top_k = min(10, len(played_idx))
    top_indices = played_idx[np.argsort(-weights)][:top_k]
    user_top_by_conf[idx2user[uid]] = top_indices
    
    # Taste diversity: std of item vectors in history
    if len(played_idx) > 1:
        hist_vecs = item_factors_normed[played_idx]
        user_taste_std[uid] = hist_vecs.std(axis=0).mean()

# User centroid DataFrame
user_centroid_df = pd.DataFrame({
    'steamid': [idx2user[i] for i in range(n_users)],
    'user_taste_diversity': user_taste_std
})

print(f"Centroids computed for {(user_centroids.sum(axis=1) != 0).sum()} users")

# ============================================================
# 3. SAVE ALL
# ============================================================
# Factor DataFrames
uf_cols = [f'uf_{i}' for i in range(N_FACTORS)]
if_cols = [f'if_{i}' for i in range(N_FACTORS)]

uf_df = pd.DataFrame(user_factors, columns=uf_cols)
uf_df['steamid'] = [idx2user[i] for i in range(n_users)]

if_df = pd.DataFrame(item_factors, columns=if_cols)
if_df['appid'] = [idx2item[i] for i in range(n_items)]

uf_df.to_parquet('artifacts/v4_user_factors.parquet', index=False)
if_df.to_parquet('artifacts/v4_item_factors.parquet', index=False)
np.save('artifacts/v4_item_factors_normed.npy', item_factors_normed)
np.save('artifacts/v4_user_centroids.npy', user_centroids)
user_centroid_df.to_parquet('artifacts/v4_user_centroid_meta.parquet', index=False)

with open('artifacts/v4_user_top_conf.pkl', 'wb') as f:
    pickle.dump(user_top_by_conf, f)
with open('artifacts/v4_user2idx.pkl', 'wb') as f:
    pickle.dump(user2idx, f)
with open('artifacts/v4_item2idx.pkl', 'wb') as f:
    pickle.dump(item2idx, f)

print("\nAll v4 ALS artifacts saved!")
print(f"Factors: {N_FACTORS}")


History: (342880, 10)
Unique users: 4564, Unique items: 18953

[1] Rebuilding ALS model and extracting latent factors...
Matrix size: 4564 users × 19237 items
Non-zeros: 342880, Density: 0.003905
Training ALS with 64 factors...


  0%|          | 0/30 [00:00<?, ?it/s]

ALS trained!
User factors shape: (4564, 64)
Item factors shape: (19237, 64)
User factors saved: (4564, 65)
Item factors saved: (19237, 65)

[2] Computing item-item co-occurrence features...
Item factors normalized for cosine similarity
Computed centroids for 4564 users
Computed top items for 4564 users

[3] Computing smoothed target encodings...
Global mean target: 3.0453
Developer TE: 11377 developers
Publisher TE: 8265 publishers
Genre combo TE: 1167 combos
User-Dev TE: 264494 pairs

All v3 features saved!


Loaded all artifacts. N_FACTORS=64

Assembling v3: artifacts/train_reranker_base.parquet
  [1] ALS base...
  [2] ALS factor cross-features...
  Computing ALS factor features...
  [3] Item-item similarity...
  Computing co-occurrence features...
  [4] User features...
  [5] Item features...
  [6] Dev/Pub cross features...
  Computing user-dev cross TE...
  [7] Interaction features...
  [8] Cleanup...
  Final shape: (1064400, 100)

Assembling v3: artifacts/test_reranker_base.parquet
  [1] ALS base...
  [2] ALS factor cross-features...
  Computing ALS factor features...
  [3] Item-item similarity...
  Computing co-occurrence features...
  [4] User features...
  [5] Item features...
  [6] Dev/Pub cross features...
  Computing user-dev cross TE...
  [7] Interaction features...
  [8] Cleanup...
  Final shape: (266100, 100)

Train v3: (1064400, 100)
Test v3:  (266100, 100)

Columns (100):
    0. steamid — int64, nulls=0
    1. appid — int64, nulls=0
    2. als_score — float64, nulls=0
    3. 

In [8]:
import pandas as pd
import numpy as np
from catboost import CatBoostRanker, Pool
from catboost.utils import get_gpu_device_count
import mlflow
import mlflow.catboost

print(f"GPU: {get_gpu_device_count()}")

train = pd.read_parquet("artifacts/train_v3.parquet").sort_values("steamid")
test = pd.read_parquet("artifacts/test_v3.parquet").sort_values("steamid")

drop_cols = ["steamid", "appid", "target"]
cat_features = ["loccountrycode", "type"]

for cf in cat_features:
    train[cf] = train[cf].fillna("unknown").astype(str)
    test[cf] = test[cf].fillna("unknown").astype(str)

feature_cols = [c for c in train.columns if c not in drop_cols]
print(f"Features: {len(feature_cols)}")

X_train, y_train, q_train = train[feature_cols], train["target"], train["steamid"]
X_test, y_test, q_test = test[feature_cols], test["target"], test["steamid"]

train_pool = Pool(data=X_train, label=y_train, group_id=q_train, cat_features=cat_features)
test_pool = Pool(data=X_test, label=y_test, group_id=q_test, cat_features=cat_features)

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("Steam_RecSys_Reranking_v3")

# ============================================================
# Experiment 1: YetiRank
# ============================================================
print("\n" + "="*60)
print("Experiment 1: YetiRank")
print("="*60)

with mlflow.start_run(run_name="v3_YetiRank"):
    params = {
        "iterations": 3000,
        "learning_rate": 0.05,
        "depth": 7,
        "l2_leaf_reg": 3.0,
        "loss_function": "YetiRank",
        "custom_metric": ["NDCG:top=10", "MAP:top=10"],
        "eval_metric": "NDCG:top=10",
        "early_stopping_rounds": 150,
        "random_seed": 42,
        "task_type": "GPU",
        "devices": "0",
        "verbose": 100,
    }

    mlflow.log_params(params)
    model_yr = CatBoostRanker(**params)
    model_yr.fit(train_pool, eval_set=test_pool, verbose=100)

    best = model_yr.get_best_score()
    ndcg_yr = best["validation"]["NDCG:top=10;type=Base"]
    print(f"\nYetiRank NDCG@10: {ndcg_yr:.4f}")
    mlflow.log_metric("best_ndcg_10", ndcg_yr)
    model_yr.save_model("artifacts/catboost_v3_yetirank.cbm")

# ============================================================
# Experiment 2: PairLogitPairwise (для сравнения)
# ============================================================
print("\n" + "="*60)
print("Experiment 2: PairLogitPairwise")
print("="*60)

with mlflow.start_run(run_name="v3_PairLogitPairwise"):
    params2 = {
        "iterations": 3000,
        "learning_rate": 0.05,
        "depth": 6,
        "l2_leaf_reg": 3.0,
        "loss_function": "PairLogitPairwise",
        "custom_metric": ["NDCG:top=10", "MAP:top=10"],
        "eval_metric": "NDCG:top=10",
        "early_stopping_rounds": 150,
        "random_seed": 42,
        "task_type": "GPU",
        "devices": "0",
        "border_count": 128,
        "bootstrap_type": "Bayesian",
        "verbose": 100,
    }

    mlflow.log_params(params2)
    model_plp = CatBoostRanker(**params2)
    model_plp.fit(train_pool, eval_set=test_pool, verbose=100)

    best2 = model_plp.get_best_score()
    ndcg_plp = best2["validation"]["NDCG:top=10;type=Base"]
    print(f"\nPairLogitPairwise NDCG@10: {ndcg_plp:.4f}")
    mlflow.log_metric("best_ndcg_10", ndcg_plp)
    model_plp.save_model("artifacts/catboost_v3_plp.cbm")

# ============================================================
# RESULTS
# ============================================================
print("\n" + "="*60)
print("RESULTS SUMMARY")
print("="*60)
print(f"v2 baseline:      NDCG@10 = 0.3804")
print(f"v3 YetiRank:      NDCG@10 = {ndcg_yr:.4f}")
print(f"v3 PairLogitPW:   NDCG@10 = {ndcg_plp:.4f}")

# Feature importance (лучшая модель)
best_model = model_yr if ndcg_yr >= ndcg_plp else model_plp
fi = best_model.get_feature_importance(train_pool)
fi_df = pd.DataFrame({'feature': feature_cols, 'importance': fi}).sort_values('importance', ascending=False)

print(f"\nTop-40 features (best model):")
print(fi_df.head(40).to_string(index=False))
fi_df.to_csv('artifacts/feature_importance_v3.csv', index=False)


GPU: 1
Features: 97


2026/05/15 22:17:44 INFO mlflow.tracking.fluent: Experiment with name 'Steam_RecSys_Reranking_v3' does not exist. Creating a new experiment.



Experiment 1: YetiRank
Groupwise loss function. OneHotMaxSize set to 10


Default metric period is 5 because MAP, NDCG is/are not implemented for GPU
Metric NDCG:type=Base is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric NDCG:top=10;type=Base is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric NDCG:top=10;type=Base is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric MAP:top=10 is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	test: 0.2408633	best: 0.2408633 (0)	total: 55.4ms	remaining: 2m 46s
100:	test: 0.3355772	best: 0.3360222 (94)	total: 3.01s	remaining: 1m 26s
200:	test: 0.3646191	best: 0.3646191 (200)	total: 5.95s	remaining: 1m 22s
300:	test: 0.3780166	best: 0.3781874 (299)	total: 8.91s	remaining: 1m 19s
400:	test: 0.3851959	best: 0.3856991 (398)	total: 11.9s	remaining: 1m 16s
500:	test: 0.3898447	best: 0.3901881 (495)	total: 14.8s	remaining: 1m 13s
600:	test: 0.3901615	best: 0.3908665 (579)	total: 17.8s	remaining: 1m 10s
700:	test: 0.3938393	best: 0.3938393 (700)	total: 20.8s	remaining: 1m 8s
800:	test: 0.3948776	best: 0.3949320 (792)	total: 23.7s	remaining: 1m 5s
900:	test: 0.3965591	best: 0.3966999 (894)	total: 26.7s	remaining: 1m 2s
1000:	test: 0.3980960	best: 0.3984393 (990)	total: 29.6s	remaining: 59.1s
1100:	test: 0.3986035	best: 0.3986114 (1099)	total: 32.6s	remaining: 56.2s
1200:	test: 0.3991922	best: 0.3993640 (1189)	total: 35.5s	remaining: 53.2s
1300:	test: 0.4007258	best: 0.4017181 (1278

Default metric period is 5 because MAP, NDCG is/are not implemented for GPU
Metric NDCG:top=10;type=Base is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric NDCG:top=10;type=Base is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric MAP:top=10 is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	test: 0.2410767	best: 0.2410767 (0)	total: 98ms	remaining: 4m 53s
100:	test: 0.3613672	best: 0.3613672 (100)	total: 4.43s	remaining: 2m 7s
200:	test: 0.3870002	best: 0.3870002 (200)	total: 8.67s	remaining: 2m
300:	test: 0.3963035	best: 0.3967056 (290)	total: 13s	remaining: 1m 56s
400:	test: 0.4002079	best: 0.4006227 (390)	total: 17.2s	remaining: 1m 51s
500:	test: 0.4046242	best: 0.4056340 (471)	total: 21.5s	remaining: 1m 47s
600:	test: 0.4049452	best: 0.4065274 (560)	total: 25.7s	remaining: 1m 42s
700:	test: 0.4082921	best: 0.4102739 (677)	total: 29.9s	remaining: 1m 38s
800:	test: 0.4102567	best: 0.4121328 (746)	total: 34.1s	remaining: 1m 33s
bestTest = 0.4121327538
bestIteration = 746
Shrink model to first 747 iterations.

PairLogitPairwise NDCG@10: 0.4121

RESULTS SUMMARY
v2 baseline:      NDCG@10 = 0.3804
v3 YetiRank:      NDCG@10 = 0.4017
v3 PairLogitPW:   NDCG@10 = 0.4121

Top-40 features (best model):
               feature  importance
             age_years    0.027764
item_p